In [ ]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [ ]:
import importlib 
# importlib.reload(sys.modules['synth_extract.utils.markdown_helpers'])
from synth_extract.utils.markdown_helpers import pdf_to_markdown, pdf_to_markdown_using_cli

Testing

In [ ]:
pdf_path = Path("../../pdfs")
pdf_files = list(pdf_path.glob("*.pdf"))

In [ ]:
pdf_files[5]

In [ ]:
markdown_path = Path("../../pdf_library")

In [ ]:
pdf_to_markdown(pdf_files[5], markdown_path, markdown_only=True)

Testing on dev set

In [ ]:
base_path = Path("../data/development_set")
arxiv_path = base_path / "arxiv"
wiley_path = base_path / "wiley"

In [ ]:
# Convert each arXiv and Wiley PDF to Markdown beside the source PDF
converted = 0
missing_pdf = []
failed = []

for source_path in (arxiv_path, wiley_path):
    if not source_path.is_dir():
        print(f"MISSING SOURCE DIRECTORY: {source_path}")
        continue

    paper_dirs = sorted(path for path in source_path.iterdir() if path.is_dir())
    for index, paper_dir in enumerate(paper_dirs, start=1):
        pdf_files = sorted(
            path
            for path in paper_dir.iterdir()
            if path.is_file() and path.suffix.lower() == ".pdf"
        )

        if not pdf_files:
            missing_pdf.append(paper_dir)
            print(f"NO PDF: {paper_dir}")
            continue

        for pdf_file in pdf_files:
            print(
                f"[{source_path.name} {index}/{len(paper_dirs)}] "
                f"Converting {pdf_file}"
            )
            try:
                pdf_to_markdown(pdf_file, markdown_only=True)
                converted += 1
            except Exception as exc:
                failed.append((pdf_file, exc))
                print(f"FAILED: {pdf_file}: {exc}")

print(f"Converted PDFs: {converted:,}")
print(f"Paper folders without a PDF: {len(missing_pdf):,}")
print(f"Failed conversions: {len(failed):,}")

creating track dbs

In [ ]:
# # Build the ArXiv Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/arxiv_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     arxiv_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'arxiv'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(arxiv_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             arxiv_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"ArXiv rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/arxiv_track.db
ArXiv rows copied: 5,388


In [ ]:
# # Build the Elsevier Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/elsevier_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     elsevier_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'elsevier'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(elsevier_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             elsevier_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful Elsevier rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/elsevier_track.db
Successful Elsevier rows copied: 489,472


In [ ]:
# # Build the Europe PMC Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/europepmc_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     europepmc_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'europepmc'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(europepmc_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             europepmc_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful Europe PMC rows copied: {copied_count:,}")

In [ ]:
# # Build the S2ORC Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/s2orc_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     s2orc_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 's2orc'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(s2orc_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             s2orc_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful S2ORC rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/s2orc_track.db
Successful S2ORC rows copied: 150,105


In [ ]:
# # Build the Springer Nature Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/springer_nature_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     springer_nature_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'springer_nature'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(springer_nature_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             springer_nature_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful Springer Nature rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/springer_nature_track.db
Successful Springer Nature rows copied: 6,103


In [ ]:
# # Build the Wiley Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/wiley_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     wiley_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'wiley'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(wiley_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             wiley_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful Wiley rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track.db
Successful Wiley rows copied: 176,784


creating dummies

In [ ]:
# # Create dummy tracking databases from the development-set folders.
# from contextlib import closing
# from pathlib import Path
# import re
# import sqlite3

# development_root = Path("../data/development_set").resolve()
# uid_pattern = re.compile(r"^ID(\d+)$")
# source_pattern = re.compile(r"^[A-Za-z0-9_]+$")

# if not development_root.is_dir():
#     raise FileNotFoundError(
#         f"Development-set directory not found: {development_root}"
#     )

# # Discover and validate every source and paper folder before writing a DB.
# source_rows = {}
# for source_dir in sorted(
#     path for path in development_root.iterdir() if path.is_dir()
# ):
#     source = source_dir.name
#     if not source_pattern.fullmatch(source):
#         raise ValueError(f"Unsafe source folder name: {source!r}")

#     rows = []
#     for paper_dir in sorted(
#         path for path in source_dir.iterdir() if path.is_dir()
#     ):
#         paper_uid = paper_dir.name
#         match = uid_pattern.fullmatch(paper_uid)
#         if match is None:
#             raise ValueError(
#                 f"Invalid paper folder in {source}: {paper_uid!r}"
#             )

#         paper_id = int(match.group(1))
#         rows.append((paper_id, paper_uid, source))

#     if not rows:
#         raise ValueError(f"No paper folders found in {source_dir}")
#     if len({row[0] for row in rows}) != len(rows):
#         raise ValueError(f"Duplicate paper IDs found in {source_dir}")
#     if len({row[1] for row in rows}) != len(rows):
#         raise ValueError(f"Duplicate paper UIDs found in {source_dir}")

#     source_rows[source] = rows

# creation_summary = []
# for source, rows in source_rows.items():
#     track_db_path = development_root / f"{source}_track.db"

#     with closing(
#         sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
#     ) as conn:
#         conn.execute("PRAGMA busy_timeout = 60000")

#         try:
#             conn.execute("BEGIN IMMEDIATE")
#             conn.execute("DROP TABLE IF EXISTS papers")
#             conn.execute(
#                 """
#                 CREATE TABLE papers (
#                     paper_id INTEGER PRIMARY KEY,
#                     paper_uid TEXT NOT NULL UNIQUE,
#                     canonical_source TEXT NOT NULL,
#                     convert_md INTEGER DEFAULT NULL
#                 )
#                 """
#             )
#             conn.executemany(
#                 """
#                 INSERT INTO papers (
#                     paper_id, paper_uid, canonical_source, convert_md
#                 )
#                 VALUES (?, ?, ?, NULL)
#                 """,
#                 rows,
#             )

#             copied_count = conn.execute(
#                 "SELECT COUNT(*) FROM papers"
#             ).fetchone()[0]
#             pending_count = conn.execute(
#                 "SELECT COUNT(*) FROM papers WHERE convert_md IS NULL"
#             ).fetchone()[0]
#             if copied_count != len(rows) or pending_count != len(rows):
#                 raise RuntimeError(
#                     f"Verification failed for {track_db_path}"
#                 )

#             conn.commit()
#         except Exception:
#             conn.rollback()
#             raise

#     creation_summary.append((source, copied_count, track_db_path))

# for source, row_count, track_db_path in creation_summary:
#     print(f"{source}: {row_count:,} rows -> {track_db_path}")

arxiv: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/arxiv_track.db
elsevier: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/elsevier_track.db
europepmc: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/europepmc_track.db
s2orc: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/s2orc_track.db
springer_nature: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/springer_nature_track.db
wiley: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/wiley_track.db
